# Import

In [1]:
from pathlib import Path
import requests
import gzip
import xml.etree.ElementTree as ET
import json
import time
from tqdm.notebook import tqdm

In [2]:
# Dictionnaire associant le nom du média (utilisé comme identifiant de fichier) 
# à l'URL de son sitemap d'index ou à une liste de sous-sitemaps explicites.
media = {
    "le_figaro": "https://sitemaps.lefigaro.fr/lefigaro.fr/articles.xml",
    "le_capital": "https://www.capital.fr/sitemap/articles.xml",
    "nice_matin": "https://www.nicematin.com/sitemap.xml",
    "le_nouvel_observateur": "https://www.nouvelobs.com/sitemap/sitemap-index-articles.xml",
    "le_journal_du_dimanche": "https://www.lejdd.fr/sitemap.xml",
    "le_monde": "https://www.lemonde.fr/sitemap_index.xml",
    "les_echos": "https://sitemap.lesechos.fr/sitemap_index.xml",
    "telerama": "https://www.telerama.fr/sitemaps/sitemap_index.php",
    "valeurs_actuelles": [
        "https://www.valeursactuelles.com/post-sitemap.xml",
        "https://www.valeursactuelles.com/post-sitemap2.xml",
        "https://www.valeursactuelles.com/post-sitemap3.xml",
    ],
}

# Liste d'URLs de sitemaps à ignorer (ex: sitemaps de catégories ou ressources non pertinentes).
to_ignore = {
    "https://www.nicematin.com/sitemap_fixed.xml",
    "https://www.nouvelobs.com/sitemap/sitemap-index-categories.xml",
}


## Partie 2

In [6]:
media = {"l_opinion": "https://www.lopinion.fr/sitemap.xml"}

HEADERS = {
    "User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) "
                  "AppleWebKit/537.36 (KHTML, like Gecko) "
                  "Chrome/126.0.0.0 Safari/537.36"
}

to_ignore = None

In [ ]:
# Répertoire de destination pour la sauvegarde des données extraites.
OUTPUT_DIR = Path("../data/urls")

# Télécharge et retourne le contenu d'un fichier XML. Gère la décompression automatique si le format est GZIP.
def read_xml(url_cible):
    raw_content = requests.get(url_cible, headers=HEADERS, timeout=60).content
    
    if not raw_content:
        raise ValueError("Contenu vide retourné par la requête.")

    # Décompression si le contenu est au format GZIP    
    if raw_content[:2] == b'\x1f\x8b':
        raw_content = gzip.decompress(raw_content)
        
    return raw_content


for media_name, sitemap in tqdm(media.items(), desc="Médias", unit="média"):

    # Création du fichier de sortie pour le média courant
    DATA_FILE = OUTPUT_DIR / f"{media_name}_articles.jsonl"
    
    # Initialisation avec les sitemaps à ignorer
    checked_sitemap = set(to_ignore or ())

    # Chargement de l'historique des traitements si le fichier de sortie existe déjà.
    if DATA_FILE.exists():
        with open(DATA_FILE, "r", encoding="utf-8") as f:
            for line in f:
                checked_sitemap.add(json.loads(line)["sitemap"])

    # Si la source est une liste, on l'utilise directement comme liste de sous-sitemaps.
    if isinstance(sitemap, list):
        sub_sitemaps = sitemap
    
    # Sinon, on télécharge le sitemap d'index pour en extraire la liste des sous-sitemaps.
    else:
        try:
            raw_xml = read_xml(sitemap)
            root = ET.fromstring(raw_xml)
            sub_sitemaps = [loc.text for loc in root.findall(".//{http://www.sitemaps.org/schemas/sitemap/0.9}loc")]
        except Exception:
            # L'index n'a pas pu être lu/parsé : on marque le média en échec et on passe au suivant.
            with open(DATA_FILE, "a", encoding="utf-8") as f:
                f.write(json.dumps({"type": "echec", "sitemap": sitemap}) + "\n")
            continue

    for sub_sitemap in tqdm(sub_sitemaps, desc=f"{media_name}", unit="sitemap", leave=False):
        
        if sub_sitemap in checked_sitemap:
            continue

        try:
            raw_xml = read_xml(sub_sitemap)
            root = ET.fromstring(raw_xml)
            urls = [loc.text for loc in root.findall(".//{http://www.sitemaps.org/schemas/sitemap/0.9}loc")]
            result = {"type": "ok", "sitemap": sub_sitemap, "urls": urls}
        except Exception:
            result = {"type": "echec", "sitemap": sub_sitemap}

        # Sauvegarde et marquage en une seule étape
        with open(DATA_FILE, "a", encoding="utf-8") as f:
            f.write(json.dumps(result) + "\n")
        checked_sitemap.add(sub_sitemap)

        time.sleep(1)

Médias:   0%|          | 0/1 [00:00<?, ?média/s]

l_opinion:   0%|          | 0/161 [00:00<?, ?sitemap/s]

# L'Express

L'Express n'expose pas de sitemap d'index : on génère nous-mêmes la liste des sous-sitemaps, un par mois, de la forme `weeks/{année}-{mois}`. Chaque sous-sitemap contient directement les URLs d'articles, donc l'extraction est la même que pour les autres médias. La boucle couvre 1990-01 → 2026-06 et reprend là où elle s'est arrêtée grâce au fichier `.jsonl`.

In [ ]:
OUTPUT_DIR = Path("../data/urls")
DATA_FILE = OUTPUT_DIR / "l_express_articles.jsonl"

periodes = [(y, m) for y in range(1990, 2027) for m in range(1, 13) if not (y == 2026 and m > 6)]

checked_sitemap = set()
if DATA_FILE.exists():
    with open(DATA_FILE, "r", encoding="utf-8") as f:
        for line in f:
            checked_sitemap.add(json.loads(line)["sitemap"])

for (year, month) in tqdm(periodes, desc="l_express", unit="sitemap"):
    sub_sitemap = f"https://www.lexpress.fr/arc/outboundfeeds/sitemap-all/weeks/{year}-{month}/?outputType=xml"

    if sub_sitemap in checked_sitemap:
        continue

    try:
        raw_xml = read_xml(sub_sitemap)
        root = ET.fromstring(raw_xml)
        urls = [loc.text for loc in root.findall(".//{http://www.sitemaps.org/schemas/sitemap/0.9}loc")]
        result = {"type": "ok", "sitemap": sub_sitemap, "urls": urls}
    except Exception:
        result = {"type": "echec", "sitemap": sub_sitemap}

    with open(DATA_FILE, "a", encoding="utf-8") as f:
        f.write(json.dumps(result) + "\n")
    checked_sitemap.add(sub_sitemap)

    time.sleep(1)


l_express:   0%|          | 0/438 [00:00<?, ?sitemap/s]

Error: FileNotFoundError: [Errno 2] No such file or directory: 'C:/Users/E.E/Desktop/STAGE/Dashboard/stage-mids/data/url/l_express_articles.jsonl'
[31m---------------------------------------------------------------------------[39m
[31mFileNotFoundError[39m                         Traceback (most recent call last)
[36mCell[39m[36m [39m[32mIn[2][39m[32m, line 26[39m
[32m     22[39m         result = {[33m"type"[39m: [33m"ok"[39m, [33m"sitemap"[39m: sub_sitemap, [33m"urls"[39m: urls}
[32m     23[39m     [38;5;28;01mexcept[39;00m Exception:
[32m     24[39m         result = {[33m"type"[39m: [33m"echec"[39m, [33m"sitemap"[39m: sub_sitemap}
[32m     25[39m 
[32m---> [39m[32m26[39m     [38;5;28;01mwith[39;00m open(DATA_FILE, [33m"a"[39m, encoding=[33m"utf-8"[39m) [38;5;28;01mas[39;00m f:
[32m     27[39m         f.write(json.dumps(result) + [33m"\n"[39m)
[32m     28[39m     checked_sitemap.add(sub_sitemap)
[32m     29[39m 

[31mFileNotFoun

In [16]:
import requests
import gzip
import xml.etree.ElementTree as ET

url = "https://www.lopinion.fr/sitemap.xml"

# User-Agent d'un vrai navigateur pour éviter le blocage anti-bot
headers = {
    "User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) "
                  "AppleWebKit/537.36 (KHTML, like Gecko) "
                  "Chrome/126.0.0.0 Safari/537.36"
}

# 1. La requête HTTP elle-même
reponse = requests.get(url, headers=headers, timeout=60)
print("Code HTTP :", reponse.status_code)
print("Content-Type :", reponse.headers.get("Content-Type"))
print("Taille du contenu :", len(reponse.content), "octets")
print("URL finale (après redirections) :", reponse.url)

raw = reponse.content

# 2. Est-ce du GZIP ?
if raw[:2] == b'\x1f\x8b':
    print("\n-> Contenu GZIP détecté, décompression...")
    raw = gzip.decompress(raw)

# 3. À quoi ressemble le début du contenu ?
print("\n--- 500 premiers caractères ---")
print(raw[:500].decode("utf-8", errors="replace"))

# 4. Tentative de parsing XML, avec l'erreur exacte
print("\n--- Test du parsing XML ---")
try:
    root = ET.fromstring(raw)
    print("XML OK, balise racine :", root.tag)
except ET.ParseError as e:
    print("ParseError :", e)


Code HTTP : 200
Content-Type : text/xml;charset=UTF-8
Taille du contenu : 18929 octets
URL finale (après redirections) : https://www.lopinion.fr/sitemap.xml

--- 500 premiers caractères ---
<?xml version='1.0' encoding='UTF-8'?><sitemapindex xmlns="http://www.sitemaps.org/schemas/sitemap/0.9" xmlns:xsi="http://www.w3.org/2001/XMLSchema-instance" xsi:schemaLocation="http://www.sitemaps.org/schemas/sitemap/0.9 https://www.sitemaps.org/schemas/sitemap/0.9/siteindex.xsd"><sitemap><loc>https://www.lopinion.fr/sitemap-201112.xml</loc><lastmod>2026-04-26T23:07:50-04:00</lastmod></sitemap><sitemap><loc>https://www.lopinion.fr/sitemap-201304.xml</loc><lastmod>2026-04-26T23:07:49-04:00</last

--- Test du parsing XML ---
XML OK, balise racine : {http://www.sitemaps.org/schemas/sitemap/0.9}sitemapindex
